### Install required packages


In [ ]:
# Dependencies are managed by requirements.txt.
# Local setup: pip install -r requirements.txt


### Required Packages

In [ ]:
import os

from crewai import Agent, Task, Crew, Process
from langchain_openai import OpenAI


In [ ]:
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
if not OPENAI_API_KEY:
    raise RuntimeError(
        "Set OPENAI_API_KEY in your environment before running this notebook. "
        "Do not paste secrets into notebook cells."
    )

RUN_CREWAI = os.getenv("RUN_CREWAI", "false").lower() == "true"


### set up the LLM

In [ ]:
llm = OpenAI(temperature=0.1, openai_api_key=OPENAI_API_KEY)


In [ ]:
# OPENAI_API_KEY is read from the environment in the setup cell.


In [ ]:
llm


### setup tools

In [ ]:
from langchain.tools import DuckDuckGoSearchRun

search_tool = DuckDuckGoSearchRun()


In [ ]:
question_to_answer = "Conduct investment analysis on Airbnb"

**Sequential**


In [ ]:
# Define your agents with roles and goals
researcher = Agent(
  role='Senior Research Analyst',
  goal=f'Conduct comprehensive research to answer the question {question_to_answer}. With a particular focus on thinking about second order effects.',
  backstory="""You work at a leading analysis firm.
  Your expertise lies in analyzing complex issues.
  You have a knack for dissecting complex data and presenting
  actionable insights.""",
  verbose=True,
  allow_delegation=True,
  llm = llm,
  tools=[
        search_tool
      ]
)

critic = Agent(
  role='Critic',
  goal='Criticize output of research analysts to ensure the content is really insightful and not generic',
  backstory="""You are a senior analyst with an IQ of 125. You care alot about research and want to ensure insights are actionable""",
  verbose=True,
  allow_delegation=False,
  llm = llm,
  tools=[]
)

writer = Agent(
  role='Research Writer',
  goal=f'Craft a compelling research report to answer the question: {question_to_answer}',
  backstory="""You are a renowned Research Strategist, known for
  your insightful and engaging research.
  You transform complex concepts into compelling reports. Include a section on potential second order effects.""",
  verbose=True,
  allow_delegation=False,
  llm = llm,
  tools=[]
)


### Tasks to perform

In [ ]:
# Create tasks for your agents
task1 = Task(
  description=f"""Conduct a comprehensive research analysis to answer the question: {question_to_answer}""",
  agent=researcher,
  expected_output=f"Conduct a comprehensive research analysis to answer the question: {question_to_answer}"  # Add the expected output field
)

In [ ]:
# Create tasks for your agents
task2 = Task(
  description="""Critique the report and insights generated""",
  agent=critic,
  expected_output="A set of criticisms and suggested improvements"  # Add the expected output field
)

In [ ]:
task3 = Task(
  description=f"""Using the insights provided, develop a research analysis to answer the question: {question_to_answer}""",
  agent=writer,
  expected_output=f"A full research report of at least 4 paragraphs research analysis answering the question: {question_to_answer}." # Added expected output
)

### Create a Crew

In [ ]:
# Instantiate your crew with a sequential process
crew = Crew(
  agents=[researcher, critic, writer],
  tasks=[task1, task2, task3],
  verbose=2, # You can set it to 1 or 2 to different logging levels,
)

In [ ]:
crew

### Kickoff the crew - let the magic happen

In [ ]:
if RUN_CREWAI:
    result = crew.kickoff()
else:
    result = None
    print("Skipping CrewAI kickoff. Set RUN_CREWAI=true to run the agent crew.")


# Heirarchical

In [ ]:
from crewai import Agent, Task, Crew, Process

# Define your agents with roles and goals
researcher = Agent(
  role='Senior Research Analyst',
  goal=f'Conduct comprehensive research to answer the question {question_to_answer}. With a particular focus on thinking about second order effects.',
  backstory="""You work at a leading analysis firm.
  Your expertise lies in analyzing complex issues.
  You have a knack for dissecting complex data and presenting
  actionable insights.""",
  verbose=True,
  allow_delegation=True,
  llm = llm,
  tools=[
        search_tool
      ]
)

critic = Agent(
  role='Critic',
  goal='Criticize output of research analysts to ensure the content is really insightful and not generic',
  backstory="""You are a senior analyst with an IQ of 125. You care alot about research and want to ensure insights are actionable""",
  verbose=True,
  allow_delegation=False,
  llm = llm,
  tools=[]
)

writer = Agent(
  role='Research Writer',
  goal=f'Craft a compelling research report to answer the question: {question_to_answer}',
  backstory="""You are a renowned Research Strategist, known for
  your insightful and engaging research.
  You transform complex concepts into compelling reports. Include a section on potential second order effects.""",
  verbose=True,
  allow_delegation=False,
  llm = llm,
  tools=[]
)

# Define your task
task = Task(
    description=f"Conduct a comprehensive research analysis to answer the question: {question_to_answer}.",
    expected_output="A short concise report with insights for an investment analyst",
)

# Define the manager agent
manager = Agent(
    role="Project Manager",
    goal="Efficiently manage the crew and ensure high-quality task completion",
    backstory="You're an experienced project manager, skilled in overseeing complex projects and guiding teams to success. Your role is to coordinate the efforts of the crew members, ensuring that each task is completed on time and to the highest standard.",
    allow_delegation=True,
)

# Instantiate your crew with a custom manager
crew = Crew(
    agents=[researcher, writer, critic],
    tasks=[task],
    manager_agent=manager,
    process=Process.hierarchical,
)

# Start the crew's work
result = crew.kickoff() if RUN_CREWAI else None
if result is None:
    print("Skipping hierarchical CrewAI kickoff. Set RUN_CREWAI=true to run it.")


In [ ]:
from IPython.display import Markdown, display

if result is not None:
    display(Markdown(result.raw))
else:
    print("CrewAI result not available because RUN_CREWAI is not enabled.")
